# Phase 2 — Deterministic Tournament Prediction

**The model's single best answer for FIFA 2026.**

Instead of 10,000 random simulations, this predicts every match deterministically:
1. **Outcome:** Most probable result (home win / draw / away win) from model probabilities
2. **Scoreline:** Most likely Poisson scoreline consistent with that outcome
3. **Bracket:** Official FIFA 2026 bracket with Annex C third-place allocation

Every run produces the **exact same result** — no randomness involved.

**Output:** Complete match-by-match predictions, group tables, and knockout bracket through to the champion.

In [1]:
import numpy as np
import pandas as pd
import joblib, json, warnings
from pathlib import Path
from scipy.stats import poisson
warnings.filterwarnings('ignore')

PROCESSED_DIR = Path('../../data/processed')
MODELS_DIR    = Path('../../models')

# Load model
bundle = joblib.load(MODELS_DIR / 'phase2_model.pkl')
xgb_model = bundle['xgb']
rf_model  = bundle['rf']
W_XGB, W_RF = bundle['w_xgb'], bundle['w_rf']
le = bundle['label_encoder']
FEATURE_COLS = bundle['feature_cols']

# Load team data
teams_2026 = pd.read_csv(PROCESSED_DIR / 'teams_2026.csv')
schedule   = pd.read_csv(PROCESSED_DIR / 'schedule_2026.csv')
elos       = pd.read_csv(PROCESSED_DIR / 'final_elos.csv').set_index('team')['final_elo'].to_dict()
confs      = pd.read_csv(PROCESSED_DIR / 'team_confederations.csv').set_index('team')['confederation'].to_dict()
fm         = pd.read_csv(PROCESSED_DIR / 'features_matrix.csv', parse_dates=['date']).sort_values('date')
team_features_df = pd.read_csv(PROCESSED_DIR / 'team_features_by_year.csv')

# Third-place allocation
with open(str(Path('../../third_place_allocation.json'))) as f:
    THIRD_PLACE_TABLE = json.load(f)
_THIRD_ALLOC_INDEX = {
    frozenset(entry['qualifying_groups']): entry['assignments']
    for entry in THIRD_PLACE_TABLE
}

print(f'Model: XGB×{W_XGB} + RF×{W_RF}, {len(FEATURE_COLS)} features')
print(f'Teams: {len(teams_2026)}, Schedule: {len(schedule)} group matches')
print(f'Third-place combos: {len(THIRD_PLACE_TABLE)}')

Model: XGB×3 + RF×1, 97 features
Teams: 48, Schedule: 72 group matches
Third-place combos: 495


## Step 1: Feature Builder & Prediction Cache

Same feature builder as the simulation notebook — batch-predict all 2,256 matchups.

In [2]:
# ── Feature helpers (copied from 04_tournament_simulation) ──

ALL_CONFS = ['UEFA','CAF','AFC','CONCACAF','CONMEBOL','OFC','UNKNOWN']

SQUAD_FEATURES = [
    'squad_avg_overall','squad_median_overall','squad_std_overall',
    'squad_top3_avg','squad_bottom5_avg',
    'gk_avg','def_avg','mid_avg','fwd_avg',
    'strongest_unit','weakest_unit',
    'squad_total_value','squad_avg_value',
    'squad_avg_age','squad_avg_potential_gap','squad_avg_caps',
    'team_pace','team_shooting','team_passing',
    'team_dribbling','team_defending','team_physic',
]

_home_form_cols = [
    'home_win_rate_5','home_avg_scored_5','home_avg_conceded_5',
    'home_pts_per_match_5','home_matches_played_5',
    'home_win_rate_10','home_avg_scored_10','home_avg_conceded_10',
    'home_pts_per_match_10','home_matches_played_10',
]
_away_form_cols = [
    'away_win_rate_5','away_avg_scored_5','away_avg_conceded_5',
    'away_pts_per_match_5','away_matches_played_5',
    'away_win_rate_10','away_avg_scored_10','away_avg_conceded_10',
    'away_pts_per_match_10','away_matches_played_10',
]
latest_home = fm.groupby('home_team')[_home_form_cols].last()
latest_away = fm.groupby('away_team')[_away_form_cols].last()
h2h_cols = ['h2h_home_win_rate','h2h_home_avg_scored','h2h_home_avg_conceded',
            'h2h_total_meetings','h2h_recent_win_rate']
h2h_lookup = fm.groupby(['home_team','away_team'])[h2h_cols].last().to_dict('index')

tf_avail_years = sorted(team_features_df['year'].unique())
squad_lookup = {}
for _, r in team_features_df.iterrows():
    squad_lookup[(r['team'], int(r['year']))] = {f: r[f] for f in SQUAD_FEATURES}
NAN_SQUAD = {f: np.nan for f in SQUAD_FEATURES}

def conf_onehot(team):
    conf = confs.get(team, 'UNKNOWN')
    return {f'conf_{c}': int(conf == c) for c in ALL_CONFS}

def get_form(team):
    if team in latest_home.index:
        row = latest_home.loc[team]
        return {
            'win_rate_5': row.get('home_win_rate_5', 0.5), 'avg_scored_5': row.get('home_avg_scored_5', 1.3),
            'avg_conceded_5': row.get('home_avg_conceded_5', 1.0), 'pts_per_match_5': row.get('home_pts_per_match_5', 1.5),
            'matches_played_5': row.get('home_matches_played_5', 5),
            'win_rate_10': row.get('home_win_rate_10', 0.5), 'avg_scored_10': row.get('home_avg_scored_10', 1.3),
            'avg_conceded_10': row.get('home_avg_conceded_10', 1.0), 'pts_per_match_10': row.get('home_pts_per_match_10', 1.5),
            'matches_played_10': row.get('home_matches_played_10', 10),
        }
    if team in latest_away.index:
        row = latest_away.loc[team]
        return {
            'win_rate_5': row.get('away_win_rate_5', 0.5), 'avg_scored_5': row.get('away_avg_scored_5', 1.0),
            'avg_conceded_5': row.get('away_avg_conceded_5', 1.3), 'pts_per_match_5': row.get('away_pts_per_match_5', 1.2),
            'matches_played_5': row.get('away_matches_played_5', 5),
            'win_rate_10': row.get('away_win_rate_10', 0.5), 'avg_scored_10': row.get('away_avg_scored_10', 1.0),
            'avg_conceded_10': row.get('away_avg_conceded_10', 1.3), 'pts_per_match_10': row.get('away_pts_per_match_10', 1.2),
            'matches_played_10': row.get('away_matches_played_10', 10),
        }
    return {'win_rate_5':0.5,'avg_scored_5':1.3,'avg_conceded_5':1.3,'pts_per_match_5':1.5,'matches_played_5':5,
            'win_rate_10':0.5,'avg_scored_10':1.3,'avg_conceded_10':1.3,'pts_per_match_10':1.5,'matches_played_10':10}

def get_h2h(home, away):
    if (home, away) in h2h_lookup:
        d = h2h_lookup[(home, away)]
        return d['h2h_home_win_rate'], d['h2h_home_avg_scored'], d['h2h_home_avg_conceded'], d['h2h_total_meetings'], d['h2h_recent_win_rate']
    return 0.5, 1.3, 1.3, 0, 0.5

def get_squad(team, year=2026):
    candidates = [y for y in tf_avail_years if y <= year]
    if not candidates: return NAN_SQUAD
    return squad_lookup.get((team, max(candidates)), NAN_SQUAD)

def build_features(home, away, neutral=True):
    home_elo = elos.get(home, 1500); away_elo = elos.get(away, 1500); elo_diff = home_elo - away_elo
    hf = get_form(home); af = get_form(away)
    h2h_hw, h2h_hs, h2h_hc, h2h_tm, h2h_rw = get_h2h(home, away)
    hc = conf_onehot(home); ac = conf_onehot(away)
    same_conf = int(confs.get(home, 'X') == confs.get(away, 'Y'))
    home_form_momentum = hf['win_rate_5'] - hf['win_rate_10']
    away_form_momentum = af['win_rate_5'] - af['win_rate_10']
    home_gdf = hf['avg_scored_5'] - hf['avg_conceded_5']
    away_gdf = af['avg_scored_5'] - af['avg_conceded_5']
    net_goal_diff = home_gdf - away_gdf
    h2h_confidence = h2h_rw * (h2h_tm / (h2h_tm + 5))
    hsq = get_squad(home); asq = get_squad(away)
    sq_diffs = {f'{f}_diff': hsq[f] - asq[f] for f in SQUAD_FEATURES}
    h_ov = hsq['squad_avg_overall']; a_ov = asq['squad_avg_overall']
    h_t3 = hsq['squad_top3_avg']; a_t3 = asq['squad_top3_avg']
    h_val = hsq['squad_total_value']; a_val = asq['squad_total_value']
    def safe(a, b, op='sub'):
        if np.isnan(a) or np.isnan(b): return np.nan
        if op == 'sub': return a - b
        if op == 'div': return a / max(b, 1)
        if op == 'logdiff': return np.log1p(a) - np.log1p(b)
        if op == 'ratio': return (a + 1) / (b + 1)
    overall_ratio = safe(h_ov, a_ov, 'div'); top3_ratio = safe(h_t3, a_t3, 'div')
    value_ratio_log = safe(h_val, a_val, 'logdiff'); value_ratio = safe(h_val, a_val, 'ratio')
    h_bal = safe(hsq['strongest_unit'], hsq['weakest_unit']); a_bal = safe(asq['strongest_unit'], asq['weakest_unit'])
    squad_balance_diff = safe(h_bal, a_bal)
    h_sg = safe(h_t3, h_ov); a_sg = safe(a_t3, a_ov); star_gap_diff = safe(h_sg, a_sg)
    h_avd = safe(hsq['fwd_avg'], asq['def_avg']); a_avd = safe(asq['fwd_avg'], hsq['def_avg'])
    h_ws = (0.6*h_ov + 0.4*h_t3) if not (np.isnan(h_ov) or np.isnan(h_t3)) else np.nan
    a_ws = (0.6*a_ov + 0.4*a_t3) if not (np.isnan(a_ov) or np.isnan(a_t3)) else np.nan
    feat = {
        'home_win_rate_5': hf['win_rate_5'], 'home_avg_scored_5': hf['avg_scored_5'],
        'home_avg_conceded_5': hf['avg_conceded_5'], 'home_pts_per_match_5': hf['pts_per_match_5'],
        'home_matches_played_5': hf['matches_played_5'],
        'home_win_rate_10': hf['win_rate_10'], 'home_avg_scored_10': hf['avg_scored_10'],
        'home_avg_conceded_10': hf['avg_conceded_10'], 'home_pts_per_match_10': hf['pts_per_match_10'],
        'home_matches_played_10': hf['matches_played_10'],
        'away_win_rate_5': af['win_rate_5'], 'away_avg_scored_5': af['avg_scored_5'],
        'away_avg_conceded_5': af['avg_conceded_5'], 'away_pts_per_match_5': af['pts_per_match_5'],
        'away_matches_played_5': af['matches_played_5'],
        'away_win_rate_10': af['win_rate_10'], 'away_avg_scored_10': af['avg_scored_10'],
        'away_avg_conceded_10': af['avg_conceded_10'], 'away_pts_per_match_10': af['pts_per_match_10'],
        'away_matches_played_10': af['matches_played_10'],
        'home_form_momentum': home_form_momentum, 'away_form_momentum': away_form_momentum,
        'home_goal_diff_form': home_gdf, 'away_goal_diff_form': away_gdf, 'net_goal_diff': net_goal_diff,
        'h2h_home_win_rate': h2h_hw, 'h2h_home_avg_scored': h2h_hs,
        'h2h_home_avg_conceded': h2h_hc, 'h2h_total_meetings': h2h_tm,
        'h2h_recent_win_rate': h2h_rw, 'h2h_confidence': h2h_confidence,
        'neutral.1': int(neutral), 'tournament_importance': 60,
        'home_conf_UEFA': hc['conf_UEFA'], 'home_conf_CAF': hc['conf_CAF'],
        'home_conf_AFC': hc['conf_AFC'], 'home_conf_CONCACAF': hc['conf_CONCACAF'],
        'home_conf_CONMEBOL': hc['conf_CONMEBOL'], 'home_conf_OFC': hc['conf_OFC'],
        'home_conf_UNKNOWN': hc['conf_UNKNOWN'],
        'away_conf_UEFA': ac['conf_UEFA'], 'away_conf_CAF': ac['conf_CAF'],
        'away_conf_AFC': ac['conf_AFC'], 'away_conf_CONCACAF': ac['conf_CONCACAF'],
        'away_conf_CONMEBOL': ac['conf_CONMEBOL'], 'away_conf_OFC': ac['conf_OFC'],
        'away_conf_UNKNOWN': ac['conf_UNKNOWN'],
        'same_confederation': same_conf,
        **sq_diffs,
        'overall_ratio': overall_ratio, 'top3_ratio': top3_ratio,
        'value_ratio_log': value_ratio_log, 'value_ratio': value_ratio,
        'squad_balance_diff': squad_balance_diff, 'star_gap_diff': star_gap_diff,
        'depth_diff': sq_diffs.get('squad_bottom5_avg_diff', np.nan),
        'squad_std_diff': sq_diffs.get('squad_std_overall_diff', np.nan),
        'home_attack_vs_def': h_avd, 'away_attack_vs_def': a_avd,
        'attack_vs_def_diff': safe(h_avd, a_avd),
        'mid_battle': sq_diffs.get('mid_avg_diff', np.nan), 'gk_diff': sq_diffs.get('gk_avg_diff', np.nan),
        'pace_diff': sq_diffs.get('team_pace_diff', np.nan), 'physic_diff': sq_diffs.get('team_physic_diff', np.nan),
        'shooting_diff': sq_diffs.get('team_shooting_diff', np.nan), 'passing_diff': sq_diffs.get('team_passing_diff', np.nan),
        'defending_diff': sq_diffs.get('team_defending_diff', np.nan), 'dribbling_diff': sq_diffs.get('team_dribbling_diff', np.nan),
        'age_diff': sq_diffs.get('squad_avg_age_diff', np.nan), 'caps_diff': sq_diffs.get('squad_avg_caps_diff', np.nan),
        'potential_gap_diff': sq_diffs.get('squad_avg_potential_gap_diff', np.nan),
        'weighted_strength_diff': safe(h_ws, a_ws),
        'elo_diff': elo_diff, 'elo_diff_sq': elo_diff**2 * np.sign(elo_diff),
        'home_elo_before': home_elo, 'away_elo_before': away_elo,
    }
    return np.array([feat.get(c, np.nan) for c in FEATURE_COLS], dtype=float)

print('Feature builder defined.')

Feature builder defined.


In [3]:
%%time
# Batch predict all 48×47 matchups + build Poisson grid
all_wc_teams = teams_2026['team'].tolist()
pairs, feat_rows = [], []
for home in all_wc_teams:
    for away in all_wc_teams:
        if home != away:
            pairs.append((home, away))
            feat_rows.append(build_features(home, away, neutral=True))

X = np.array(feat_rows)
X_rf = np.nan_to_num(X, 0)
xgb_probs = xgb_model.predict_proba(X)
rf_probs  = rf_model.predict_proba(X_rf)
blended   = (W_XGB * xgb_probs + W_RF * rf_probs) / (W_XGB + W_RF)

# Poisson grid for scoreline fitting
GRID_STEP, GRID_MAX, MAX_G = 0.1, 4.0, 8
grid_vals = np.arange(0.1, GRID_MAX + GRID_STEP, GRID_STEP)
goals = np.arange(MAX_G + 1)
pmf_cache = np.array([poisson.pmf(goals, lam) for lam in grid_vals])
all_mats = np.einsum('ig,jh->ijgh', pmf_cache, pmf_cache)
g = MAX_G + 1
grid_ph = np.sum(all_mats * np.tril(np.ones((g,g), dtype=bool), -1), axis=(2,3))
grid_pd = np.sum(all_mats * np.eye(g, dtype=bool), axis=(2,3))
grid_pa = np.sum(all_mats * np.triu(np.ones((g,g), dtype=bool), 1), axis=(2,3))
del all_mats

def fit_lambdas(p_h, p_d, p_a):
    err = (grid_ph - p_h)**2 + (grid_pd - p_d)**2 + (grid_pa - p_a)**2
    idx = np.unravel_index(np.argmin(err), err.shape)
    return float(grid_vals[idx[0]]), float(grid_vals[idx[1]])

# Build caches
prob_cache = {}
lambda_cache = {}
for i, (home, away) in enumerate(pairs):
    p_a = float(blended[i, 0])
    p_d = float(blended[i, 1])
    p_h = float(blended[i, 2])
    prob_cache[(home, away)] = (p_h, p_d, p_a)
    lambda_cache[(home, away)] = fit_lambdas(p_h, p_d, p_a)

print(f'Cached {len(prob_cache)} matchups with probabilities and Poisson lambdas.')

Cached 2256 matchups with probabilities and Poisson lambdas.
CPU times: total: 2.66 s
Wall time: 1.31 s


## Step 2: Deterministic Match Predictor

For each match:
1. **Outcome** = argmax(p_home, p_draw, p_away)
2. **Scoreline** = most probable Poisson scoreline consistent with that outcome
3. **KO draws** = decided by higher win probability (simulating penalty edge)

In [4]:
def best_scoreline(lam_h, lam_a, outcome):
    """Find the most probable Poisson scoreline consistent with the outcome."""
    best_prob, best_score = -1, (0, 0)
    for hg in range(MAX_G + 1):
        for ag in range(MAX_G + 1):
            if outcome == 'home' and hg <= ag: continue
            if outcome == 'away' and ag <= hg: continue
            if outcome == 'draw' and hg != ag: continue
            p = poisson.pmf(hg, lam_h) * poisson.pmf(ag, lam_a)
            if p > best_prob:
                best_prob = p
                best_score = (hg, ag)
    return best_score

def predict_group_match(home, away):
    """Deterministic group match: most likely outcome + most likely scoreline."""
    p_h, p_d, p_a = prob_cache[(home, away)]
    lam_h, lam_a = lambda_cache[(home, away)]
    
    # Most probable outcome
    outcome = ['home', 'draw', 'away'][np.argmax([p_h, p_d, p_a])]
    hg, ag = best_scoreline(lam_h, lam_a, outcome)
    
    if outcome == 'home':   h_pts, a_pts = 3, 0
    elif outcome == 'away': h_pts, a_pts = 0, 3
    else:                   h_pts, a_pts = 1, 1
    
    return {
        'home': home, 'away': away,
        'home_goals': hg, 'away_goals': ag,
        'home_pts': h_pts, 'away_pts': a_pts,
        'outcome': outcome,
        'prob_home': round(p_h, 3), 'prob_draw': round(p_d, 3), 'prob_away': round(p_a, 3),
    }

def predict_ko_match(t1, t2, round_name):
    """Deterministic knockout match: most likely winner, scoreline, penalties if draw."""
    p_h, p_d, p_a = prob_cache[(t1, t2)]
    lam_h, lam_a = lambda_cache[(t1, t2)]
    
    outcome = ['home', 'draw', 'away'][np.argmax([p_h, p_d, p_a])]
    hg, ag = best_scoreline(lam_h, lam_a, outcome)
    
    penalties = False
    if outcome == 'draw':
        # Penalties — team with higher win probability advances
        winner = t1 if p_h >= p_a else t2
        penalties = True
    elif outcome == 'home':
        winner = t1
    else:
        winner = t2
    
    return {
        'round': round_name,
        'home': t1, 'away': t2,
        'home_goals': hg, 'away_goals': ag,
        'winner': winner, 'penalties': penalties,
        'prob_home': round(p_h, 3), 'prob_draw': round(p_d, 3), 'prob_away': round(p_a, 3),
    }

# Test
result = predict_group_match('Spain', 'Qatar')
print(f"Test: {result['home']} {result['home_goals']}-{result['away_goals']} {result['away']}  (H={result['prob_home']} D={result['prob_draw']} A={result['prob_away']})")

result = predict_ko_match('Argentina', 'France', 'Final')
print(f"Test KO: {result['home']} {result['home_goals']}-{result['away_goals']} {result['away']}  → winner: {result['winner']} (pen={result['penalties']})")

Test: Spain 2-0 Qatar  (H=0.856 D=0.109 A=0.035)
Test KO: Argentina 1-0 France  → winner: Argentina (pen=False)


## Step 3: Group Stage — All 72 Matches

In [5]:
# Predict all 72 group matches
all_group_matches = []
group_tables = {}

for group in sorted(teams_2026['group'].unique()):
    group_teams = teams_2026[teams_2026['group'] == group]['team'].tolist()
    group_matches = schedule[schedule['group'] == group]
    
    # Initialize standings
    stats = {t: {'team': t, 'pts': 0, 'gd': 0, 'gf': 0, 'ga': 0, 'w': 0, 'd': 0, 'l': 0, 'mp': 0}
             for t in group_teams}
    
    for _, row in group_matches.iterrows():
        result = predict_group_match(row['home_team'], row['away_team'])
        result['group'] = group
        all_group_matches.append(result)
        
        h, a = result['home'], result['away']
        hg, ag = result['home_goals'], result['away_goals']
        
        stats[h]['mp'] += 1; stats[a]['mp'] += 1
        stats[h]['gf'] += hg; stats[h]['ga'] += ag
        stats[a]['gf'] += ag; stats[a]['ga'] += hg
        stats[h]['gd'] += hg - ag; stats[a]['gd'] += ag - hg
        stats[h]['pts'] += result['home_pts']; stats[a]['pts'] += result['away_pts']
        
        if result['outcome'] == 'home':
            stats[h]['w'] += 1; stats[a]['l'] += 1
        elif result['outcome'] == 'away':
            stats[a]['w'] += 1; stats[h]['l'] += 1
        else:
            stats[h]['d'] += 1; stats[a]['d'] += 1
    
    standing = sorted(stats.values(), key=lambda x: (x['pts'], x['gd'], x['gf']), reverse=True)
    group_tables[group] = standing

# Display all groups
print('=' * 80)
print('GROUP STAGE PREDICTIONS — MODEL\'S BEST ANSWER')
print('=' * 80)

for group in sorted(group_tables.keys()):
    print(f'\n── Group {group} ──')
    standing = group_tables[group]
    print(f'  {"Team":<22} {"MP":>3} {"W":>3} {"D":>3} {"L":>3} {"GF":>3} {"GA":>3} {"GD":>4} {"PTS":>4}')
    for i, s in enumerate(standing):
        marker = '★' if i < 2 else ('△' if i == 2 else ' ')
        print(f'{marker} {s["team"]:<22} {s["mp"]:>3} {s["w"]:>3} {s["d"]:>3} {s["l"]:>3} '
              f'{s["gf"]:>3} {s["ga"]:>3} {s["gd"]:>+4} {s["pts"]:>4}')
    
    # Show matches
    group_results = [m for m in all_group_matches if m['group'] == group]
    for m in group_results:
        print(f'    {m["home"]:>20} {m["home_goals"]}-{m["away_goals"]} {m["away"]:<20}  '
              f'({m["prob_home"]:.0%} / {m["prob_draw"]:.0%} / {m["prob_away"]:.0%})')

print(f'\n\nTotal group matches: {len(all_group_matches)}')

GROUP STAGE PREDICTIONS — MODEL'S BEST ANSWER

── Group A ──
  Team                    MP   W   D   L  GF  GA   GD  PTS
★ Mexico                   3   3   0   0   4   0   +4    9
★ South Korea              3   1   1   1   1   1   +0    4
△ Czech Republic           3   1   1   1   1   1   +0    4
  South Africa             3   0   0   3   0   4   -4    0
                  Mexico 2-0 South Africa          (82% / 15% / 3%)
             South Korea 0-0 Czech Republic        (36% / 41% / 23%)
          Czech Republic 1-0 South Africa          (67% / 22% / 12%)
                  Mexico 1-0 South Korea           (50% / 31% / 19%)
          Czech Republic 0-1 Mexico                (36% / 27% / 37%)
            South Africa 0-1 South Korea           (22% / 26% / 52%)

── Group B ──
  Team                    MP   W   D   L  GF  GA   GD  PTS
★ Switzerland              3   3   0   0   4   0   +4    9
★ Canada                   3   2   0   1   3   1   +2    6
△ Bosnia and Herzegovina   3   1   0   

## Step 4: Knockout Stage — R32 through Final

Official FIFA 2026 bracket with Annex C third-place allocation.

In [6]:
# Extract group results
winners = {g: group_tables[g][0]['team'] for g in group_tables}
runners_up = {g: group_tables[g][1]['team'] for g in group_tables}

# Best 8 third-place teams
third_place = []
for g in group_tables:
    s = group_tables[g][2]
    third_place.append({'team': s['team'], 'pts': s['pts'], 'gd': s['gd'], 'gf': s['gf'], 'group': g})
third_sorted = sorted(third_place, key=lambda x: (x['pts'], x['gd'], x['gf']), reverse=True)
best_thirds = {d['group']: d['team'] for d in third_sorted[:8]}
eliminated_thirds = [d for d in third_sorted[8:]]

print('GROUP WINNERS:')
for g in sorted(winners): print(f'  {g}: {winners[g]}')
print('\nRUNNERS-UP:')
for g in sorted(runners_up): print(f'  {g}: {runners_up[g]}')
print('\nBEST 8 THIRD-PLACE (qualify for R32):')
for d in third_sorted[:8]: print(f'  Group {d["group"]}: {d["team"]} ({d["pts"]}pts, {d["gd"]:+}gd)')
print('\nELIMINATED THIRD-PLACE:')
for d in eliminated_thirds: print(f'  Group {d["group"]}: {d["team"]} ({d["pts"]}pts, {d["gd"]:+}gd)')

# Third-place allocation lookup
def _lookup_third_allocation(qualifying_groups):
    key = frozenset(qualifying_groups)
    return _THIRD_ALLOC_INDEX[key]

# Build R32 bracket
alloc = _lookup_third_allocation(list(best_thirds.keys()))
print(f'\nThird-place allocation for groups {sorted(best_thirds.keys())}:')
for match, group in sorted(alloc.items()):
    print(f'  Match {match}: 3rd from Group {group} ({best_thirds[group]})')

GROUP WINNERS:
  A: Mexico
  B: Switzerland
  C: Brazil
  D: Turkey
  E: Germany
  F: Netherlands
  G: Belgium
  H: Spain
  I: France
  J: Argentina
  K: Colombia
  L: England

RUNNERS-UP:
  A: South Korea
  B: Canada
  C: Scotland
  D: United States
  E: Ecuador
  F: Japan
  G: Egypt
  H: Uruguay
  I: Norway
  J: Algeria
  K: Portugal
  L: Croatia

BEST 8 THIRD-PLACE (qualify for R32):
  Group A: Czech Republic (4pts, +0gd)
  Group J: Austria (3pts, +0gd)
  Group I: Senegal (3pts, -1gd)
  Group L: Ghana (3pts, -1gd)
  Group B: Bosnia and Herzegovina (3pts, -1gd)
  Group C: Morocco (3pts, -1gd)
  Group F: Sweden (3pts, -1gd)
  Group K: DR Congo (3pts, -1gd)

ELIMINATED THIRD-PLACE:
  Group D: Paraguay (3pts, -2gd)
  Group E: Curaçao (3pts, -2gd)
  Group G: Iran (3pts, -2gd)
  Group H: Cape Verde (3pts, -3gd)

Third-place allocation for groups ['A', 'B', 'C', 'F', 'I', 'J', 'K', 'L']:
  Match 74: 3rd from Group C (Morocco)
  Match 77: 3rd from Group F (Sweden)
  Match 79: 3rd from Group

In [7]:
# ── Play entire knockout stage ──

# Official FIFA bracket constants
R16_PAIRINGS = [(74, 77), (73, 75), (76, 78), (79, 80), (83, 84), (81, 82), (86, 88), (85, 87)]
QF_PAIRINGS  = [(0, 1), (4, 5), (2, 3), (6, 7)]
SF_PAIRINGS  = [(0, 1), (2, 3)]

all_ko_matches = []

# ── R32: Matches 73-88 ──
r32 = {}
r32[73] = (runners_up['A'], runners_up['B'])
r32[75] = (winners['F'], runners_up['C'])
r32[76] = (winners['C'], runners_up['F'])
r32[78] = (runners_up['E'], runners_up['I'])
r32[83] = (runners_up['K'], runners_up['L'])
r32[84] = (winners['H'], runners_up['J'])
r32[86] = (winners['J'], runners_up['H'])
r32[88] = (runners_up['D'], runners_up['G'])
r32[74] = (winners['E'], best_thirds[alloc['74']])
r32[77] = (winners['I'], best_thirds[alloc['77']])
r32[79] = (winners['A'], best_thirds[alloc['79']])
r32[80] = (winners['L'], best_thirds[alloc['80']])
r32[81] = (winners['D'], best_thirds[alloc['81']])
r32[82] = (winners['G'], best_thirds[alloc['82']])
r32[85] = (winners['B'], best_thirds[alloc['85']])
r32[87] = (winners['K'], best_thirds[alloc['87']])

print('=' * 80)
print('KNOCKOUT STAGE PREDICTIONS')
print('=' * 80)

print('\n── ROUND OF 32 ──')
r32_winners = {}
for match_num in sorted(r32.keys()):
    t1, t2 = r32[match_num]
    result = predict_ko_match(t1, t2, 'R32')
    result['match_num'] = match_num
    all_ko_matches.append(result)
    r32_winners[match_num] = result['winner']
    pen = ' (PEN)' if result['penalties'] else ''
    print(f'  M{match_num}: {t1:>20} {result["home_goals"]}-{result["away_goals"]} {t2:<20} → {result["winner"]}{pen}  '
          f'({result["prob_home"]:.0%}/{result["prob_draw"]:.0%}/{result["prob_away"]:.0%})')

# ── R16 ──
print('\n── ROUND OF 16 ──')
r16_winners = []
r16_match_num = 89
for m_a, m_b in R16_PAIRINGS:
    t1, t2 = r32_winners[m_a], r32_winners[m_b]
    result = predict_ko_match(t1, t2, 'R16')
    result['match_num'] = r16_match_num
    all_ko_matches.append(result)
    r16_winners.append(result['winner'])
    pen = ' (PEN)' if result['penalties'] else ''
    print(f'  M{r16_match_num}: {t1:>20} {result["home_goals"]}-{result["away_goals"]} {t2:<20} → {result["winner"]}{pen}  '
          f'({result["prob_home"]:.0%}/{result["prob_draw"]:.0%}/{result["prob_away"]:.0%})')
    r16_match_num += 1

# ── QF ──
print('\n── QUARTER-FINALS ──')
qf_winners = []
qf_match_num = 97
for i_a, i_b in QF_PAIRINGS:
    t1, t2 = r16_winners[i_a], r16_winners[i_b]
    result = predict_ko_match(t1, t2, 'QF')
    result['match_num'] = qf_match_num
    all_ko_matches.append(result)
    qf_winners.append(result['winner'])
    pen = ' (PEN)' if result['penalties'] else ''
    print(f'  M{qf_match_num}: {t1:>20} {result["home_goals"]}-{result["away_goals"]} {t2:<20} → {result["winner"]}{pen}  '
          f'({result["prob_home"]:.0%}/{result["prob_draw"]:.0%}/{result["prob_away"]:.0%})')
    qf_match_num += 1

# ── SF ──
print('\n── SEMI-FINALS ──')
sf_winners = []
sf_losers = []
sf_match_num = 101
for i_a, i_b in SF_PAIRINGS:
    t1, t2 = qf_winners[i_a], qf_winners[i_b]
    result = predict_ko_match(t1, t2, 'SF')
    result['match_num'] = sf_match_num
    all_ko_matches.append(result)
    sf_winners.append(result['winner'])
    sf_losers.append(t1 if result['winner'] == t2 else t2)
    pen = ' (PEN)' if result['penalties'] else ''
    print(f'  M{sf_match_num}: {t1:>20} {result["home_goals"]}-{result["away_goals"]} {t2:<20} → {result["winner"]}{pen}  '
          f'({result["prob_home"]:.0%}/{result["prob_draw"]:.0%}/{result["prob_away"]:.0%})')
    sf_match_num += 1

# ── Third Place Match ──
print('\n── THIRD PLACE MATCH ──')
t1, t2 = sf_losers[0], sf_losers[1]
result_3rd = predict_ko_match(t1, t2, '3rd Place')
result_3rd['match_num'] = 103
all_ko_matches.append(result_3rd)
pen = ' (PEN)' if result_3rd['penalties'] else ''
print(f'  M103: {t1:>20} {result_3rd["home_goals"]}-{result_3rd["away_goals"]} {t2:<20} → {result_3rd["winner"]}{pen}')

# ── Final ──
print('\n── FINAL ──')
t1, t2 = sf_winners[0], sf_winners[1]
result_final = predict_ko_match(t1, t2, 'Final')
result_final['match_num'] = 104
all_ko_matches.append(result_final)
pen = ' (PEN)' if result_final['penalties'] else ''
print(f'  M104: {t1:>20} {result_final["home_goals"]}-{result_final["away_goals"]} {t2:<20} → {result_final["winner"]}{pen}')

champion = result_final['winner']
runner_up = t1 if champion == t2 else t2
third = result_3rd['winner']
fourth = sf_losers[0] if third == sf_losers[1] else sf_losers[1]

print('\n' + '=' * 80)
print(f'🏆 PREDICTED CHAMPION: {champion.upper()}')
print(f'🥈 Runner-up: {runner_up}')
print(f'🥉 Third: {third}')
print(f'   Fourth: {fourth}')
print('=' * 80)

KNOCKOUT STAGE PREDICTIONS

── ROUND OF 32 ──
  M73:          South Korea 1-0 Canada               → South Korea  (42%/32%/25%)
  M74:              Germany 1-0 Morocco              → Germany  (62%/23%/15%)
  M75:          Netherlands 1-0 Scotland             → Netherlands  (73%/18%/8%)
  M76:               Brazil 1-0 Japan                → Brazil  (61%/23%/16%)
  M77:               France 1-0 Sweden               → France  (75%/19%/6%)
  M78:              Ecuador 1-0 Norway               → Ecuador  (37%/33%/29%)
  M79:               Mexico 1-0 Senegal              → Mexico  (40%/33%/27%)
  M80:              England 1-0 DR Congo             → England  (76%/18%/6%)
  M81:               Turkey 1-0 Bosnia and Herzegovina → Turkey  (74%/19%/7%)
  M82:              Belgium 2-0 Czech Republic       → Belgium  (63%/20%/17%)
  M83:             Portugal 2-1 Croatia              → Portugal  (59%/21%/20%)
  M84:                Spain 2-0 Algeria              → Spain  (82%/14%/4%)
  M85:          Sw

## Step 5: Save All Results

In [8]:
# Save group matches
group_matches_df = pd.DataFrame(all_group_matches)
group_matches_df.to_csv(PROCESSED_DIR / 'deterministic_group_matches.csv', index=False)

# Save group tables
table_rows = []
for group in sorted(group_tables.keys()):
    for pos, s in enumerate(group_tables[group], 1):
        s_copy = s.copy()
        s_copy['group'] = group
        s_copy['position'] = pos
        s_copy['qualifies'] = 'winner' if pos == 1 else ('runner_up' if pos == 2 else ('3rd_qualified' if s['team'] in best_thirds.values() else 'eliminated'))
        table_rows.append(s_copy)
group_tables_df = pd.DataFrame(table_rows)
group_tables_df.to_csv(PROCESSED_DIR / 'deterministic_group_tables.csv', index=False)

# Save knockout matches
ko_matches_df = pd.DataFrame(all_ko_matches)
ko_matches_df.to_csv(PROCESSED_DIR / 'deterministic_ko_matches.csv', index=False)

# Save final summary
summary = {
    'champion': champion,
    'runner_up': runner_up,
    'third': third,
    'fourth': fourth,
    'sf_winners': sf_winners,
    'sf_losers': sf_losers,
    'qf_winners': qf_winners,
    'group_winners': winners,
    'runners_up': runners_up,
    'best_thirds': best_thirds,
    'third_place_allocation': alloc,
}
with open(PROCESSED_DIR / 'deterministic_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print('Saved:')
print(f'  deterministic_group_matches.csv   ({len(group_matches_df)} matches)')
print(f'  deterministic_group_tables.csv    ({len(group_tables_df)} rows, 12 groups)')
print(f'  deterministic_ko_matches.csv      ({len(ko_matches_df)} matches)')
print(f'  deterministic_summary.json        (champion: {champion})')
print()
print(f'Total matches predicted: {len(all_group_matches) + len(all_ko_matches)}')

Saved:
  deterministic_group_matches.csv   (72 matches)
  deterministic_group_tables.csv    (48 rows, 12 groups)
  deterministic_ko_matches.csv      (32 matches)
  deterministic_summary.json        (champion: France)

Total matches predicted: 104
